In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 23. KV Cache — 추론 속도의 핵심 [CPU/GPU 벤치마크 ⑩]

> **학습 목표**
> - KV Cache가 왜 추론 속도를 $O(n^2) \to O(n)$으로 낮추는지 이해한다
> - Prefill vs Decode 단계를 구분한다
> - 캐시 메모리 계산법을 안다
> - KV Cache 사용 시/미사용 시 속도를 비교한다

## 23.1 추론 단계: Prefill vs Decode

LLM 추론은 두 단계:
1. **Prefill (Context Encoding)**: 프롬프트 전체 처리. 병렬 가능. 연산량 $O(n^2)$.
2. **Decode (Token Generation)**: 토큰을 하나씩 생성. 순차적. 매 스텝 $O(n)$.

Decode가 전체 시간의 대부분. KV Cache는 decode 속도를 크게 개선.

## 23.2 왜 K, V만 캐시하는가

어텐션: $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$

토큰 $t$ 생성 시:
- $Q_t$ (새 쿼리) 필요
- $K_{\leq t}, V_{\leq t}$ (이전 모든 키, 값) 필요

캐시 없으면 매 스텝 모든 이전 토큰의 $K, V$를 재계산 → $O(t^2)$.
캐시 있으면 $K_{\leq t-1}, V_{\leq t-1}$ 저장해두고 $K_t, V_t$만 추가 → $O(t)$.

**Q는 캐시 안 함**: 매 스텝 새 $Q_t$ 하나만 필요.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

# 캐시 없는 어텐션 (매 스텝 전체 재계산)
def attention_no_cache(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# 캐시 있는 어텐션 (증분)
def attention_with_cache(q_new, k_cache, v_cache, k_new, v_new):
    """q_new: (B, 1, d), k_cache/v_cache: (B, T, d)"""
    # Cache에 새 k, v 추가
    k_full = torch.cat([k_cache, k_new], dim=1)  # (B, T+1, d)
    v_full = torch.cat([v_cache, v_new], dim=1)
    d_k = q_new.shape[-1]
    scores = q_new @ k_full.transpose(-1, -2) / (d_k ** 0.5)  # (B, 1, T+1)
    weights = F.softmax(scores, dim=-1)
    out = weights @ v_full  # (B, 1, d)
    return out, k_full, v_full

# Comparison: 100 토큰 Generation
np.random.seed(0)
torch.manual_seed(0)
B, d, T = 1, 64, 1
n_generate = 50

# Cache 없는 방식
t0 = time.perf_counter()
Q = torch.randn(B, T, d)
K = torch.randn(B, T, d)
V = torch.randn(B, T, d)
for step in range(n_generate):
    # 매 Step 전체 재Computation
    Q_new = torch.randn(B, 1, d)
    Q = torch.cat([Q, Q_new], dim=1)
    K = torch.cat([K, torch.randn(B, 1, d)], dim=1)
    V = torch.cat([V, torch.randn(B, 1, d)], dim=1)
    out = attention_no_cache(Q, K, V)
t_no_cache = time.perf_counter() - t0
print(f"캐시 없음: {t_no_cache*1000:.2f}ms ({n_generate} 토큰)")

# Cache 있는 방식
torch.manual_seed(0)
t0 = time.perf_counter()
k_cache = torch.randn(B, 1, d)
v_cache = torch.randn(B, 1, d)
for step in range(n_generate):
    q_new = torch.randn(B, 1, d)
    k_new = torch.randn(B, 1, d)
    v_new = torch.randn(B, 1, d)
    out, k_cache, v_cache = attention_with_cache(q_new, k_cache, v_cache, k_new, v_new)
t_cache = time.perf_counter() - t0
print(f"캐시 사용: {t_cache*1000:.2f}ms ({n_generate} 토큰)")
print(f"Speed Improvement: {t_no_cache/t_cache:.2f}x")


## 23.3 KV Cache 메모리 계산

KV Cache 메모리 = $2 \cdot L \cdot h \cdot d_k \cdot n \cdot \text{dtype}$

- $L$: 레이어 수
- $h$: KV 헤드 수 (MHA면 $h$, MQA면 1, GQA면 $g$)
- $d_k$: 헤드 차원
- $n$: 시퀀스 길이
- 2: K와 V
- dtype: FP16=2 bytes, FP32=4 bytes

예: LLaMA-7B ($L=32, h=32, d_k=128$), FP16, $n=4096$:
$$2 \times 32 \times 32 \times 128 \times 4096 \times 2 = 2 \text{ GB}$$

$n=32768$이면 16GB → GPU 메모리 대부분 차지.


In [ ]:
# KV Cache 메모리 계산기
def kv_cache_memory_gb(n_layers, n_kv_heads, d_k, seq_len, dtype_bytes=2):
    """KV Cache Memory (GB)."""
    return 2 * n_layers * n_kv_heads * d_k * seq_len * dtype_bytes / (1024**3)

# 다양한 Model/Sequence Length
models = [
    ('LLaMA-7B', 32, 32, 128, 2),
    ('LLaMA-13B', 40, 40, 128, 2),
    ('LLaMA-70B', 80, 8, 128, 2),  # GQA
    ('GPT-2 small', 12, 12, 64, 2),
    ('Mistral-7B', 32, 8, 128, 2),  # GQA
]
seq_lens = [2048, 8192, 32768, 131072]

print(f"{'Model':>15} | ", end='')
for sl in seq_lens:
    print(f"n={sl:>6}", end=' | ')
print()
print("-" * 75)
for name, L, h, dk, dt in models:
    print(f"{name:>15} | ", end='')
    for sl in seq_lens:
        mem = kv_cache_memory_gb(L, h, dk, sl, dt)
        print(f"{mem:>7.2f}GB", end=' | ')
    print()

print("\n=> Sequence Length가 길어지Plane KV Cache가 Memory 대Subspace을 차지한다.")
print("=> GQA(Mistral, LLaMA-70B)는 KV Head 수가 적어 Memory Efficiency적.")


## 23.4 [CPU/GPU 벤치마크 ⑩] KV Cache on/off


In [ ]:
# KV Cache 벤치마크
from llm_math.bench import time_fn

# 단일 어텐션 층에서 캐시 효과 측정
def bench_decode_no_cache(n_context, d=64, n_decode=20):
    """Cache 없이 n_decode 토큰 Generation."""
    Q = torch.randn(1, n_context, d)
    K = torch.randn(1, n_context, d)
    V = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        Q_new = torch.randn(1, 1, d)
        Q = torch.cat([Q, Q_new], dim=1)
        K = torch.cat([K, torch.randn(1, 1, d)], dim=1)
        V = torch.cat([V, torch.randn(1, 1, d)], dim=1)
        _ = Q @ K.transpose(-1, -2) / (d ** 0.5) @ V

def bench_decode_with_cache(n_context, d=64, n_decode=20):
    """Cache 사용하여 n_decode 토큰 Generation."""
    k_cache = torch.randn(1, n_context, d)
    v_cache = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        q_new = torch.randn(1, 1, d)
        k_new = torch.randn(1, 1, d)
        v_new = torch.randn(1, 1, d)
        k_full = torch.cat([k_cache, k_new], dim=1)
        v_full = torch.cat([v_cache, v_new], dim=1)
        _ = q_new @ k_full.transpose(-1, -2) / (d ** 0.5) @ v_full
        k_cache = k_full
        v_cache = v_full

print(f"{'n_context':>10} | {'No Cache (ms)':>15} | {'Cache (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n_ctx in [128, 512, 1024, 2048]:
    t_no = time_fn(bench_decode_no_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    t_yes = time_fn(bench_decode_with_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    print(f"{n_ctx:>10} | {t_no:>15.3f} | {t_yes:>12.3f} | {t_no/t_yes:>9.2f}x")
print("\n=> Context가 길수록 KV Cache의 Effect가 크다 (O(n) vs O(n^2)).")


## 23.5 Continuous Batching

고전적 batching: 배치 내 모든 시퀀스가 동시에 끝나야 함 → 짧은 시퀀스가 긴 것을 기다림.

**Continuous Batching** (vLLM 등): 각 시퀀스가 끝나면 즉시 새 요청으로 교체.
- 처리량(thoughtput) 크게 향상
- 가변 길이 요청 처리 가능

## 23.6 PagedAttention (vLLM)

OS 페이징에서 영감:
- KV Cache를 고정 크기 "페이지"로 분할
- 비연속적 메모리에 저장
- 페이지 테이블로 관리

메모리 단편화 해결, 큰 배치 처리 가능. vLLM이 2-4x 빠른 이유.

## 23.7 핵심 정리

| 개념 | 의미 |
|---|---|
| Prefill | 프롬프트 병렬 처리, $O(n^2)$ |
| Decode | 토큰 생성, $O(n)$ per token (KV cache) |
| KV Cache | $K, V$ 저장, 매 스텝 $K_t, V_t$만 추가 |
| 메모리 | $2 L h d_k n \cdot \text{dtype}$ |
| Continuous Batching | 가변 길이, 높은 처리량 |
| PagedAttention | 메모리 단편화 해결 |

## 연습문제

1. KV Cache on/off로 100토큰 생성 시간을 n=128, 512, 2048에서 비교하라.
2. LLaMA-7B에서 컨텍스트 32K의 KV Cache 메모리를 계산하라.
3. GQA가 KV Cache 메모리를 얼마나 절약하는지 LLaMA-7B(32 헤드) vs LLaMA-70B(8 KV 헤드)로 비교하라.
4. Continuous Batching의 처리량 향상을 시뮬레이션으로 보이라.
5. PagedAttention이 메모리 단편화를 어떻게 해결하는지 설명하라.

> 해설: `solutions/ch23_solutions.ipynb`
